# 🌊 YOLO-UDD v2.0 - Underwater Debris Detection Training

Complete training notebook for Google Colab with GPU support.

**Repository:** https://github.com/kshitijkhede/YOLO-UDD-v2.0

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0
print("✓ Repository cloned!")

## Step 2: Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations opencv-python-headless tqdm pyyaml tensorboard
print("✓ Dependencies installed!")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted!")

## Step 4: Configure Training

In [ ]:
# ⚠️ IMPORTANT: Set your Google Drive dataset path
# Your dataset structure:
#   TrashCan_dataset/
#     ├── original_data/
#     │   ├── images/
#     │   └── annotations/
#     └── scripts/
#         └── trash_can_coco.py

DATASET_PATH = '/content/drive/MyDrive/TrashCan_dataset/original_data'

# Training settings
BATCH_SIZE = 8
EPOCHS = 50
LEARNING_RATE = 0.01
SAVE_DIR = 'runs/train'

print(f"Dataset: {DATASET_PATH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"\n⚠️  Make sure your dataset has:")
print(f"   - {DATASET_PATH}/images/")
print(f"   - {DATASET_PATH}/annotations/")

## Step 5: Check GPU & Dataset

In [ ]:
import torch
import os
import json

print("GPU Check:")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

print("\nDataset Check:")
if os.path.exists(DATASET_PATH):
    print(f"  ✓ Path exists: {DATASET_PATH}")
    
    # Check for original_data structure (individual JSON files per image)
    images_dir = os.path.join(DATASET_PATH, 'images')
    annotations_dir = os.path.join(DATASET_PATH, 'annotations')
    
    if os.path.exists(images_dir):
        image_count = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
        print(f"  ✓ Images folder: {image_count} images found")
    else:
        print(f"  ❌ Images folder not found")
        
    if os.path.exists(annotations_dir):
        ann_count = len([f for f in os.listdir(annotations_dir) if f.endswith('.json')])
        print(f"  ✓ Annotations folder: {ann_count} annotation files found")
    else:
        print(f"  ❌ Annotations folder not found")
    
    # Check for COCO format files
    train_coco = os.path.join(DATASET_PATH, 'instances_train_trashcan.json')
    val_coco = os.path.join(DATASET_PATH, 'instances_val_trashcan.json')
    
    if os.path.exists(train_coco):
        with open(train_coco, 'r') as f:
            train_data = json.load(f)
            print(f"  ✓ Train COCO: {len(train_data.get('images', []))} images")
    else:
        print(f"  ℹ️  No COCO train file - will need conversion")
        
    if os.path.exists(val_coco):
        with open(val_coco, 'r') as f:
            val_data = json.load(f)
            print(f"  ✓ Val COCO: {len(val_data.get('images', []))} images")
    else:
        print(f"  ℹ️  No COCO val file - will need conversion")
else:
    print(f"  ❌ Path not found: {DATASET_PATH}")
    print("  Please update DATASET_PATH in Step 4!")

## Step 6: Train Model

In [ ]:
!python scripts/train.py \
  --data-dir "{DATASET_PATH}" \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --lr {LEARNING_RATE} \
  --save-dir {SAVE_DIR}

## Step 7: View Results

In [ ]:
import glob

checkpoints = glob.glob(f'{SAVE_DIR}/checkpoints/*.pt')
if checkpoints:
    print(f"✓ Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        size = os.path.getsize(ckpt) / (1024*1024)
        print(f"  📦 {os.path.basename(ckpt)} ({size:.1f} MB)")
else:
    print("❌ No checkpoints found")

## Step 8: Download Best Model

In [ ]:
from google.colab import files

best_model = f'{SAVE_DIR}/checkpoints/best_model.pt'
if os.path.exists(best_model):
    files.download(best_model)
    print("✓ Download started!")
else:
    print("❌ Model not found")